# Doc Processor Categorization Demo

This notebook runs the `doc_processor` LangGraph parser, converts paragraph parser metadata into `TextAnnotation` objects, and renders a **Parser Category Review** HTML preview.

By the end you should have:

1. A parsed `DocIR` with `paragraph.meta.parser` category metadata.
2. A category count summary and color legend.
3. A saved HTML review file under `tests/results/doc_processor_categorization_demo/`.
4. An inline iframe preview of the highlighted document.
    


## 1. Setup

Run this notebook from either the repository root or `apps/backend/doc_processor`. The path detection below finds the local package and sample documents.
    


In [1]:
from __future__ import annotations

from collections import Counter
from html import escape
from pathlib import Path
import sys
from uuid import uuid4

from IPython.display import HTML, display
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


def find_doc_processor_project(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "doc_processor").exists():
            return candidate
        nested = candidate / "apps" / "backend" / "doc_processor"
        if (nested / "pyproject.toml").exists() and (nested / "src" / "doc_processor").exists():
            return nested
    raise RuntimeError("Could not find apps/backend/doc_processor project directory.")


PROJECT_DIR = find_doc_processor_project(Path.cwd().resolve())
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

TESTS_DIR = PROJECT_DIR / "tests"
DOC_SAMPLES = TESTS_DIR / "doc_samples" / "Korean_Labor_Subcontract_Risk_Benchmark_Pack"
OUTPUT_DIR = TESTS_DIR / "results" / "doc_processor_categorization_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from doc_processor import (
    ContractReviewConfig,
    ContractReviewGraphState,
    ParagraphCategory,
    ParserConfig,
    ReviewContractRequest,
    RelevanceMode,
    build_contract_review_clients_from_env,
    build_contract_review_graph,
    check_contract_review_env,
    run_parser,
)
from doc_processor.api import DocumentInput, TextAnnotation, apply_document_edits, render_review_html


def show_html(html: str, *, height: int = 760) -> None:
    display(
        HTML(
            f'<iframe srcdoc="{escape(html, quote=True)}" '
            f'width="100%" height="{height}" '
            'style="border:1px solid #d0d7de; border-radius:6px; background:#fff;"></iframe>'
        )
    )


PROJECT_DIR, DOC_SAMPLES, OUTPUT_DIR


(PosixPath('/home/maxjo/Work/LAS-system/apps/backend/doc_processor'),
 PosixPath('/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/Korean_Labor_Subcontract_Risk_Benchmark_Pack'),
 PosixPath('/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/results/doc_processor_categorization_demo'))

## 2. Choose a Sample Document

The default sample is an HWPX contract. Change `SAMPLE_DOC` to another file in `tests/doc_samples/new_test/` if you want to compare parser behavior.
    


In [2]:
SAMPLE_DOC = DOC_SAMPLES / "04_HDC-04_소프트웨어_개발용역_하도급계약서.docx"
# SAMPLE_DOC = DOC_SAMPLES / "01_KLC-01_정규직_매장운영매니저_근로계약서.docx"
# SAMPLE_DOC = DOC_SAMPLES / "02. 청소년 대중문화예술인 표준 부속합의서.hwpx"
# SAMPLE_DOC = DOC_SAMPLES / "style_test_sample.docx"

assert SAMPLE_DOC.exists(), SAMPLE_DOC
SAMPLE_DOC
    

PosixPath('/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/Korean_Labor_Subcontract_Risk_Benchmark_Pack/04_HDC-04_소프트웨어_개발용역_하도급계약서.docx')

## 3. Run The LangGraph Parser

`run_parser(...)` builds and invokes the parser graph. `USE_LLM_REVIEW` controls the boundary and label LLM review steps; keep it `True` when credentials are configured, or set it to `False` for a local no-LLM parser pass.


In [3]:
USE_LLM_REVIEW = True
PARSER_MAX_CONCURRENT_WORKERS = 2

parser_state = run_parser(
    SAMPLE_DOC,
    config=ParserConfig(
        relevance_mode=RelevanceMode.KEYWORD_THEN_LLM if USE_LLM_REVIEW else RelevanceMode.DISABLED,
        boundary_review_enabled=USE_LLM_REVIEW,
        label_review_enabled=USE_LLM_REVIEW,
        max_concurrent_workers=PARSER_MAX_CONCURRENT_WORKERS,
        langfuse_enabled=True,
    ),
)

parser_doc = parser_state.working_doc
parser_result = parser_state.parser_result
assert parser_doc is not None
assert parser_result is not None

{
    "accepted": parser_result.accepted,
    "reason": parser_result.reason,
    "clause_count": parser_result.clause_count,
    "subclause_count": parser_result.subclause_count,
    "notes": list(parser_result.notes),
}
    

2026-05-08 15:07:49,804 | INFO | structure analysis run start source=/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/Korean_Labor_Subcontract_Risk_Benchmark_Pack/04_HDC-04_소프트웨어_개발용역_하도급계약서.docx
2026-05-08 15:07:49,980 | INFO | [structure_analysis.load_document] start
2026-05-08 15:07:50,070 | INFO | [structure_analysis.load_document] done goto=screen_relevance
2026-05-08 15:07:50,077 | INFO | [structure_analysis.screen_relevance] start
2026-05-08 15:07:50,078 | INFO | [structure_analysis.screen_relevance] done goto=regex_analysis
2026-05-08 15:07:50,080 | INFO | [structure_analysis.regex_analysis] start
2026-05-08 15:07:50,081 | INFO | [structure_analysis.regex_analysis] done goto=llm_analysis
2026-05-08 15:07:50,084 | INFO | [structure_analysis.llm_analysis] start
2026-05-08 15:07:50,085 | INFO | [structure_analysis.llm_analysis] dispatching boundary batch review (6 suspects)
2026-05-08 15:07:50,085 | INFO | [structure_analysis.llm_analysis] done goto=boundar

{'accepted': True,
 'reason': 'Parser clause parsing completed.',
 'clause_count': 12,
 'subclause_count': 32,
 'notes': ["Detected clause rule 'article' and subclause rule 'numeric_dot'."]}

## 4. Convert Parser Metadata To Review Annotations

The parser writes per-paragraph metadata to `paragraph.meta.parser`. This cell maps those categories/spans into `TextAnnotation` payloads that the HTML renderer can highlight.
    


In [4]:
PARSER_CATEGORY_COLORS = {
    ParagraphCategory.TITLE: "#D9EAD3",
    ParagraphCategory.PREAMBLE: "#FFF2CC",
    ParagraphCategory.CLAUSE_HEADING: "#F4CCCC",
    ParagraphCategory.CLAUSE_BODY: "#FCE5CD",
    ParagraphCategory.SUBCLAUSE_HEADING: "#D9D2E9",
    ParagraphCategory.SUBCLAUSE_BODY: "#CFE2F3",
    ParagraphCategory.INPUT_BLOCK: "#D0E0E3",
    ParagraphCategory.APPENDIX: "#EAD1DC",
    ParagraphCategory.HEADER: "#EFEFEF",
    ParagraphCategory.FOOTER: "#EFEFEF",
    ParagraphCategory.OTHER: "#E6E6E6",
    ParagraphCategory.BOUNDARY_SUSPECT: "#F9CB9C",
}


def parser_category_label(category: ParagraphCategory | None) -> str:
    if category is None:
        return "Unlabeled"
    return category.value.replace("_", " ").title()


def parser_category_color(category: ParagraphCategory | None) -> str:
    if category is None:
        return "#E6E6E6"
    return PARSER_CATEGORY_COLORS.get(category, "#E6E6E6")


def occurrence_index_for_span(text: str, selected_text: str, start: int) -> int | None:
    matches = []
    search_from = 0
    while True:
        index = text.find(selected_text, search_from)
        if index < 0:
            break
        matches.append(index)
        search_from = index + 1
    if not matches:
        raise ValueError(f"Could not find {selected_text!r} in paragraph text.")
    occurrence_index = matches.index(start)
    return occurrence_index if len(matches) > 1 else None


def note_from_parser_meta(parser_meta, *, category: ParagraphCategory | None = None) -> str:
    note_parts = []
    if category is not None:
        note_parts.append(f"category={category.value}")
    if parser_meta.clause_no:
        note_parts.append(f"clause={parser_meta.clause_no}")
    if parser_meta.subclause_no:
        note_parts.append(f"subclause={parser_meta.subclause_no}")
    if parser_meta.boundary_suspect:
        note_parts.append("boundary_suspect=true")
    return " | ".join(note_parts)


def parser_annotations_from_doc(doc):
    annotations = []
    skipped = []
    for paragraph in doc.paragraphs:
        parser_meta = paragraph.meta.parser if paragraph.meta and paragraph.meta.parser else None
        if parser_meta is None or not paragraph.text.strip():
            continue
        if paragraph.tables or paragraph.images:
            skipped.append(paragraph.node_id)
            continue

        if parser_meta.spans:
            for span in parser_meta.spans:
                selected_text = paragraph.text[span.start:span.end]
                if not selected_text:
                    continue
                annotations.append(
                    TextAnnotation(
                        target_kind="paragraph",
                        target_id=paragraph.node_id,
                        selected_text=selected_text,
                        occurrence_index=occurrence_index_for_span(paragraph.text, selected_text, span.start),
                        label=parser_category_label(span.kind),
                        color=parser_category_color(span.kind),
                        note=note_from_parser_meta(parser_meta, category=span.kind),
                    )
                )
            continue

        if parser_meta.category is None:
            continue
        annotations.append(
            TextAnnotation(
                target_kind="paragraph",
                target_id=paragraph.node_id,
                label=parser_category_label(parser_meta.category),
                color=parser_category_color(parser_meta.category),
                note=note_from_parser_meta(parser_meta, category=parser_meta.category),
            )
        )
    return annotations, skipped


parser_annotations, skipped_mixed_content = parser_annotations_from_doc(parser_doc)
category_counts = Counter(annotation.label for annotation in parser_annotations)

{
    "annotation_count": len(parser_annotations),
    "category_counts": dict(sorted(category_counts.items())),
    "skipped_mixed_content_paragraph_count": len(skipped_mixed_content),
    "skipped_mixed_content_examples": skipped_mixed_content[:10],
}
    

{'annotation_count': 49,
 'category_counts': {'Clause Heading': 12,
  'Footer': 1,
  'Preamble': 2,
  'Subclause Body': 1,
  'Subclause Heading': 32,
  'Title': 1},
 'skipped_mixed_content_paragraph_count': 2,
 'skipped_mixed_content_examples': ['p_6b2ed4bd498de58d',
  'p_98fa1dd5264e7946']}

## 5. Display The Category Legend

The legend uses the same colors as the annotations that will be rendered into the document preview.
    


In [5]:
def category_legend_html(counts: Counter) -> str:
    rows = []
    for category, color in PARSER_CATEGORY_COLORS.items():
        label = parser_category_label(category)
        rows.append(
            "<tr>"
            f"<td><span style='display:inline-block;width:14px;height:14px;background:{color};border:1px solid #888;'></span></td>"
            f"<td style='padding:4px 10px 4px 6px;'>{label}</td>"
            f"<td style='text-align:right;padding:4px 0;'>{counts.get(label, 0)}</td>"
            "</tr>"
        )
    return """
    <div style='font-family:system-ui, sans-serif; max-width:480px;'>
      <h3 style='margin:0 0 8px;'>Parser Category Legend</h3>
      <table style='border-collapse:collapse;'>%s</table>
    </div>
    """ % "".join(rows)


display(HTML(category_legend_html(category_counts)))
    

,Title,1
,Preamble,2
,Clause Heading,12
,Clause Body,0
,Subclause Heading,32
,Subclause Body,1
,Input Block,0
,Appendix,0
,Header,0
,Footer,1
,Other,0


## 6. Render And Display The Parser Category Review

`render_review_html(...)` resolves each annotation against the current `DocIR`. The resulting HTML is saved and displayed inline using an iframe.
    


In [6]:
review = render_review_html(
    document=DocumentInput(doc_ir=parser_doc),
    annotations=parser_annotations,
    title="Parser Category Review",
)

review_path = OUTPUT_DIR / f"{SAMPLE_DOC.stem}_parser_category_review.html"
review_path.write_text(review.html or "", encoding="utf-8")

{
    "ok": review.ok,
    "review_path": str(review_path.relative_to(PROJECT_DIR)),
    "resolved_annotation_count": len(review.resolved_annotations),
    "validation_issues": [issue.model_dump(mode="json") for issue in review.validation.issues],
}
    

{'ok': True,
 'review_path': 'tests/results/doc_processor_categorization_demo/04_HDC-04_소프트웨어_개발용역_하도급계약서_parser_category_review.html',
 'resolved_annotation_count': 49,
 'validation_issues': []}

In [7]:
if review.ok and review.html:
    show_html(review.html, height=820)
else:
    display([issue.model_dump(mode="json") for issue in review.validation.issues])
    

/home/maxjo/Work/LAS-system/apps/backend/doc_processor/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 7. Combined Contract Review Graph (RAG + HITL)

This graph runs document loading, parser categorization, clause-risk review fanout, result aggregation, and optional HITL pause in one LangGraph workflow. The RAG and model calls stay opt-in through the env check.


In [8]:
RUN_RISK_ANALYSIS = True
RISK_MAX_CLAUSES = 15
PARSER_MAX_CONCURRENT_WORKERS = 3
RISK_MAX_CONCURRENT_REVIEWS = 3
RISK_PROVIDER_RETRY_ATTEMPTS = 4
RISK_PROVIDER_RETRY_BASE_DELAY_SEC = 2.0
PAUSE_FOR_HITL = True
HITL_MIN_RISK_LEVEL = "low"  # change to "mid", "high", or "crit" to reduce review noise
HITL_DEFAULT_ACTION = "accept"  # "accept"로 바꾸면 전체 수정안을 수락합니다
RESUME_HITL_WITH_DEMO_DECISIONS = False

risk_env_status = check_contract_review_env()
risk_env_status.model_dump(mode="json")


{'ready': True,
 'missing': [],
 'warnings': [],
 'qdrant_url_configured': True,
 'qdrant_collections': ['law_article', 'legal_case', 'legal_relation'],
 'opensearch_url_configured': False,
 'opensearch_indices': [],
 'embedding_model': 'text-embedding-3-large',
 'embedding_dimensions': 1024,
 'llm_provider': 'gemini',
 'llm_model': 'gemini-2.5-flash-lite'}

In [9]:
def first_interrupt_payload(raw_graph_state):
    interrupts = raw_graph_state.get("__interrupt__") if isinstance(raw_graph_state, dict) else None
    if not interrupts:
        return None
    return interrupts[0].value


HITL_SEVERITY_STYLES = {
    "none": {"accent": "#8c959f", "badge_bg": "#f6f8fa", "badge_text": "#24292f", "badge_border": "#d0d7de", "card_bg": "#ffffff"},
    "low": {"accent": "#1a7f37", "badge_bg": "#dafbe1", "badge_text": "#0f5132", "badge_border": "#4ac26b", "card_bg": "#fbfffb"},
    "mid": {"accent": "#9a6700", "badge_bg": "#fff8c5", "badge_text": "#3b2300", "badge_border": "#d4a72c", "card_bg": "#fffdf2"},
    "high": {"accent": "#bc4c00", "badge_bg": "#ffebe9", "badge_text": "#7d2d00", "badge_border": "#fb8f44", "card_bg": "#fff8f4"},
    "crit": {"accent": "#cf222e", "badge_bg": "#ffebe9", "badge_text": "#82071e", "badge_border": "#ff8182", "card_bg": "#fff5f5"},
}


def hitl_requests_html(requests) -> str:
    if not requests:
        return "<p>No HITL requests.</p>"
    cards = []
    for request in requests:
        diff = request.get("diff") or ""
        risk = str(request.get("risk_level", "none") or "none").lower()
        style = HITL_SEVERITY_STYLES.get(risk, HITL_SEVERITY_STYLES["mid"])
        citations = request.get("source_citations") or []
        citation_html = "".join(f"<li>{escape(str(citation))}</li>" for citation in citations[:5])
        if citation_html:
            citation_html = f'<ul style="margin:8px 0 0 18px;color:#24292f;">{citation_html}</ul>'
        cards.append(
            """
            <section style="border:1px solid #8c959f;border-left:6px solid {accent};border-radius:6px;padding:14px;margin:12px 0;background:{card_bg};color:#1f2328;box-shadow:0 1px 2px rgba(31,35,40,0.08);">
              <div style="display:flex;gap:8px;align-items:center;flex-wrap:wrap;">
                <span style="background:{badge_bg};color:{badge_text};border:1px solid {badge_border};border-radius:999px;padding:2px 8px;font-weight:700;text-transform:uppercase;">{risk}</span>
                <strong>{title}</strong>
              </div>
              <div style="margin-top:6px;color:#57606a;font-size:13px;">{kind} · finding_id={finding_id}</div>
              <p style="margin:10px 0;color:#1f2328;">{prompt}</p>
              <p style="margin:10px 0;color:#1f2328;"><strong>Guidance</strong><br>{guidance}</p>
              <pre style="white-space:pre-wrap;background:#f6f8fa;color:#1f2328;border:1px solid #d0d7de;border-radius:6px;padding:10px;overflow:auto;">{diff}</pre>
              {citations}
            </section>
            """.format(
                accent=style["accent"],
                card_bg=style["card_bg"],
                badge_bg=style["badge_bg"],
                badge_text=style["badge_text"],
                badge_border=style["badge_border"],
                risk=escape(risk),
                title=escape(str(request.get("title", ""))),
                kind=escape(str(request.get("kind", ""))),
                finding_id=escape(str(request.get("finding_id", ""))),
                prompt=escape(str(request.get("prompt", ""))),
                guidance=escape(str(request.get("guidance", ""))),
                diff=escape(diff),
                citations=citation_html,
            )
        )
    return "".join(cards)


In [10]:
risk_graph = None
risk_graph_state = None
risk_state = None
risk_result = None
hitl_payload = None
risk_thread_config = None

if RUN_RISK_ANALYSIS:
    if not risk_env_status.ready:
        raise RuntimeError(f"RAG env is incomplete: {risk_env_status.missing}")

    clients = build_contract_review_clients_from_env()
    risk_graph = build_contract_review_graph(checkpointer=InMemorySaver())
    risk_thread_config = {
        "configurable": {
            "thread_id": f"doc-review-demo-{SAMPLE_DOC.stem}-{uuid4().hex[:8]}",
            "rag_client": clients.rag_client,
            "generation_client": clients.generation_client,
        },
        "max_concurrency": RISK_MAX_CONCURRENT_REVIEWS,
    }
    review_request = ReviewContractRequest(
        source_path=str(SAMPLE_DOC),
        relevance_mode=RelevanceMode.KEYWORD_THEN_LLM if USE_LLM_REVIEW else RelevanceMode.DISABLED,
        boundary_review_enabled=USE_LLM_REVIEW,
        label_review_enabled=USE_LLM_REVIEW,
        parser_max_concurrent_workers=PARSER_MAX_CONCURRENT_WORKERS,
        config=ContractReviewConfig(
            max_clauses=RISK_MAX_CLAUSES,
            top_k=8,
            include_review_html=True,
            review_title="계약 리스크 검토",
            pause_for_hitl=PAUSE_FOR_HITL,
            hitl_min_risk_level=HITL_MIN_RISK_LEVEL,
            max_concurrent_risk_reviews=RISK_MAX_CONCURRENT_REVIEWS,
            max_generation_provider_retry_attempts=RISK_PROVIDER_RETRY_ATTEMPTS,
            generation_provider_retry_base_delay_sec=RISK_PROVIDER_RETRY_BASE_DELAY_SEC,
        ),
    )
    risk_graph_state = risk_graph.invoke(
        ContractReviewGraphState(request=review_request).model_dump(mode="json"),
        config=risk_thread_config,
    )
    hitl_payload = first_interrupt_payload(risk_graph_state)
    risk_state = ContractReviewGraphState.model_validate(risk_graph_state)
    risk_result = risk_state.result

    risk_review_path = OUTPUT_DIR / f"{SAMPLE_DOC.stem}_risk_review_graph.html"
    risk_review_path.write_text(risk_result.review_html or "", encoding="utf-8")

    display({
        "paused_for_hitl": hitl_payload is not None,
        "overall_risk_level": risk_result.risk_level if risk_result else None,
        "clause_risk_counts": risk_result.clause_risk_counts if risk_result else None,
        "finding_count": len(risk_result.findings) if risk_result else 0,
        "hitl_request_count": len(risk_result.hitl_requests) if risk_result else 0,
        "review_path": str(risk_review_path.relative_to(PROJECT_DIR)),
        "warnings": risk_result.warnings if risk_result else [],
    })
else:
    display({
        "skipped": True,
        "reason": "Set RUN_RISK_ANALYSIS = True after configuring RAG env.",
        "env_ready": risk_env_status.ready,
        "missing": risk_env_status.missing,
        "warnings": risk_env_status.warnings,
    })


2026-05-08 15:07:54,887 | INFO | structure analysis run start source=/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/Korean_Labor_Subcontract_Risk_Benchmark_Pack/04_HDC-04_소프트웨어_개발용역_하도급계약서.docx
2026-05-08 15:07:54,890 | INFO | [structure_analysis.load_document] start
2026-05-08 15:07:54,950 | INFO | [structure_analysis.load_document] done goto=screen_relevance
2026-05-08 15:07:54,959 | INFO | [structure_analysis.screen_relevance] start
2026-05-08 15:07:54,964 | INFO | [structure_analysis.screen_relevance] done goto=regex_analysis
2026-05-08 15:07:54,966 | INFO | [structure_analysis.regex_analysis] start
2026-05-08 15:07:54,970 | INFO | [structure_analysis.regex_analysis] done goto=llm_analysis
2026-05-08 15:07:54,973 | INFO | [structure_analysis.llm_analysis] start
2026-05-08 15:07:54,975 | INFO | [structure_analysis.llm_analysis] dispatching boundary batch review (6 suspects)
2026-05-08 15:07:54,975 | INFO | [structure_analysis.llm_analysis] done goto=boundar

Contract review generation validation failed; retrying clause review (clause_id=clause-3, clause_no=3, attempt=1/3, issues=finding 1: source_ids must include at least one supplied source_id.; finding 2: source_ids must include at least one supplied source_id.)
Contract review generation validation failed; retrying clause review (clause_id=clause-5, clause_no=5, attempt=1/3, issues=finding 1: selected_text must be an exact substring of target_node_id paragraph text.; finding 1: replacement_text cannot be applied because selected_text is not in the target paragraph.; finding 1: proposed edit changes the target paragraph numbering from '1' to '2'.; finding 1: proposed edit contains another top-level numbered subclause.)
Contract review generation validation failed; retrying clause review (clause_id=clause-7, clause_no=7, attempt=1/3, issues=finding 1: source_ids must include at least one supplied source_id.; finding 2: source_ids must include at least one supplied source_id.)
Contract rev

{'paused_for_hitl': True,
 'overall_risk_level': 'high',
 'clause_risk_counts': {'none': 4, 'low': 6, 'mid': 0, 'high': 2, 'crit': 0},
 'finding_count': 12,
 'hitl_request_count': 12,
 'review_path': 'tests/results/doc_processor_categorization_demo/04_HDC-04_소프트웨어_개발용역_하도급계약서_risk_review_graph.html',
 'warnings': ['Review generation repaired after 2 attempts.',
  'Review generation repaired after 2 attempts.',
  'Review generation validation still failed after 3 attempts; falling back to normalized output. Issues: finding 1: source_ids must include at least one supplied source_id.',
  'Review generation validation still failed after 3 attempts; falling back to normalized output. Issues: finding 1: source_ids must include at least one supplied source_id.',
  'Review generation validation still failed after 3 attempts; falling back to normalized output. Issues: finding 2: source_ids must include at least one supplied source_id.',
  'Review generation repaired after 2 attempts.']}

In [11]:
if risk_result is not None and risk_result.review_html:
    show_html(risk_result.review_html, height=820)
elif risk_result is not None:
    display([finding.model_dump(mode="json") for finding in risk_result.findings])


In [12]:
if hitl_payload:
    display(HTML(hitl_requests_html(hitl_payload["requests"])))

In [13]:
hitl_payload

{'type': 'contract_review_hitl',
 'summary': {'overall_risk_level': 'high',
  'finding_count': 12,
  'request_count': 12},
 'requests': [{'request_id': 'clause-3:finding:1:hitl',
   'finding_id': 'clause-3:finding:1',
   'clause_id': 'clause-3',
   'clause_no': '3',
   'risk_level': 'high',
   'title': '수급사업자의 작업 거절권 박탈',
   'kind': 'suggested_edit',
   'prompt': '제안된 수정안을 적용하기 전에 검토하세요.',
   'guidance': '긴급한 변경사항에 대한 작업 거절권을 제한하더라도, 수급사업자가 최소한의 이의를 제기하거나 추후 정산을 통해 권리를 구제받을 수 있는 방안을 마련해야 합니다. 예를 들어, 작업 거절은 불가하나, 변경된 내용에 대한 이의 제기 또는 추가적인 협의 기회를 보장하는 내용을 추가하는 것을 고려할 수 있습니다.',
   'selected_text': '서면이 늦게 작성되거나 생략되어도 수급사업자는 해당 작업을 거절할 수 없다',
   'proposed_edit': {'edit_type': 'text',
    'client_edit_id': None,
    'target_kind': 'paragraph',
    'target_id': 'p_4b464a6bb30d2b7b',
    'expected_text_hash': '114ed14d99b31d69d0403c4c117c33c31ba9adc4',
    'new_text': '3. 다만, 긴급 패치, 신규 API 필드 추가, 고객사 요청사항 반영 등은 원사업자의 프로젝트 관리도구 이슈 등록 또는 메신저 지시만으로 즉시 효력이 발생하며, 세부 서면은 작업 착수 후 또는 월말 정산 시 작성할 수 있다.

In [14]:
from collections import Counter

edit_conflicts = [k 
                  for k, v 
                  in Counter([p['proposed_edit']['target_id'] 
                              for p in 
                              hitl_payload['requests'] 
                              if p['proposed_edit'] is not None]
                             ).items() if v > 1]
edit_conflicts

['p_52fa12b5f1fa8ace']

In [15]:
if hitl_payload:
    request_ids = [request["finding_id"] for request in hitl_payload["requests"]]
    existing_decisions = {
        decision.get("finding_id"): decision
        for decision in globals().get("HITL_DECISIONS", [])
        if isinstance(decision, dict) and decision.get("finding_id") in request_ids
    }
    should_rebuild_decisions = set(existing_decisions) != set(request_ids)
    if should_rebuild_decisions:
        HITL_DECISIONS = [
            {
                "finding_id": request["finding_id"],
                "action": HITL_DEFAULT_ACTION,
                "comment": "노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.",
            }
            for request in hitl_payload["requests"]
        ]
    else:
        HITL_DECISIONS = [existing_decisions[finding_id] for finding_id in request_ids]

    display({
        "next_step": "HITL_DECISIONS를 확인/수정하고 RESUME_HITL_WITH_DEMO_DECISIONS = True로 바꾼 뒤 이 셀을 다시 실행하세요.",
        "actions": ["accept", "reject", "feedback", "provide_info", "manual_edit"],
        "decision_action_counts": dict(Counter(decision.get("action") for decision in HITL_DECISIONS)),
        "accepted_decision_count": sum(decision.get("action") == "accept" for decision in HITL_DECISIONS),
    })

    # 노트북용 HITL: 전체 수락은 위 설정의 HITL_DEFAULT_ACTION = "accept"로 바꾸고
    # 리스크 그래프 셀부터 다시 실행하세요. 일부만 수락하려면 아래 리스트의 action을 직접 수정한 뒤
    # RESUME_HITL_WITH_DEMO_DECISIONS = True로 바꾸고 이 셀을 실행합니다.
    display(HITL_DECISIONS)

    if RESUME_HITL_WITH_DEMO_DECISIONS:
        risk_graph_state = risk_graph.invoke(
            Command(resume={"decisions": HITL_DECISIONS}),
            config=risk_thread_config,
        )
        risk_state = ContractReviewGraphState.model_validate(risk_graph_state)
        risk_result = risk_state.result
        requested_actions = {decision["finding_id"]: decision.get("action") for decision in HITL_DECISIONS}
        applied_actions = {decision.finding_id: decision.action for decision in risk_result.human_decisions}
        action_mismatches = {
            finding_id: {"requested": requested, "applied": applied_actions.get(finding_id)}
            for finding_id, requested in requested_actions.items()
            if applied_actions.get(finding_id) not in {None, requested}
        }
        display({
            "resumed": True,
            "requested_action_counts": dict(Counter(requested_actions.values())),
            "applied_action_counts": dict(Counter(applied_actions.values())),
            "status_counts": dict(Counter(finding.status for finding in risk_result.findings)),
            "accepted_status_count": sum(finding.status == "accepted" for finding in risk_result.findings),
            "action_mismatches": action_mismatches,
            "note": "action_mismatches가 비어있지 않다면 이미 다른 결정으로 종료된 thread입니다. 리스크 그래프 셀을 다시 실행해 새 thread를 만든 뒤 HITL 셀을 다시 실행하세요.",
        })
elif RUN_RISK_ANALYSIS:
    display({"paused_for_hitl": False, "reason": "No source-backed findings met the HITL threshold."})


{'next_step': 'HITL_DECISIONS를 확인/수정하고 RESUME_HITL_WITH_DEMO_DECISIONS = True로 바꾼 뒤 이 셀을 다시 실행하세요.',
 'actions': ['accept', 'reject', 'feedback', 'provide_info', 'manual_edit'],
 'decision_action_counts': {'accept': 12},
 'accepted_decision_count': 12}

[{'finding_id': 'clause-3:finding:1',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'clause-3:finding:2',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'clause-5:finding:1',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'clause-6:finding:3',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'clause-7:finding:1',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'clause-7:finding:2',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'clause-8:finding:1',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'clause-9:finding:1',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'clause-9:finding:2',
  'action': 'accept',
  'comment': '노트북 데모: 법무/사업 검토 의견 대기 상태로 둡니다.'},
 {'finding_id': 'cl

In [16]:
if risk_result is not None:
    display([
        {
            "risk_level": finding.risk_level,
            "status": finding.status,
            "title": finding.title,
            "clause_no": finding.clause_no,
            "target_node_ids": finding.target_node_ids,
            "hitl_kind": finding.human_request.kind if finding.human_request else None,
            "sources": [source.citation or source.source_id for source in finding.sources],
            "recommendation": finding.recommendation,
        }
        for finding in risk_result.findings[:10]
    ])


[{'risk_level': 'high',
  'status': 'pending',
  'title': '수급사업자의 작업 거절권 박탈',
  'clause_no': '3',
  'target_node_ids': ['p_4b464a6bb30d2b7b'],
  'hitl_kind': 'suggested_edit',
  'sources': ['하도급거래 공정화에 관한 법률 제3조 (law::001590::article::3::0)',
   '하도급거래 공정화에 관한 법률 시행령 제3조 (law::005367::article::3::0)'],
  'recommendation': '긴급한 변경사항에 대한 작업 거절권을 제한하더라도, 수급사업자가 최소한의 이의를 제기하거나 추후 정산을 통해 권리를 구제받을 수 있는 방안을 마련해야 합니다. 예를 들어, 작업 거절은 불가하나, 변경된 내용에 대한 이의 제기 또는 추가적인 협의 기회를 보장하는 내용을 추가하는 것을 고려할 수 있습니다.'},
 {'risk_level': 'low',
  'status': 'pending',
  'title': '변경 합의서 작성 시점의 모호성',
  'clause_no': '3',
  'target_node_ids': ['p_67e6166b1b5c0b03'],
  'hitl_kind': 'suggested_edit',
  'sources': ['하도급거래 공정화에 관한 법률 제3조 (law::001590::article::3::0)'],
  'recommendation': "계약 범위 변경 시 변경합의서 작성 시점을 명확히 규정하고, 제3항의 긴급 변경 시 서면 생략 규정과 충돌하지 않도록 조정하는 것이 좋습니다. 예를 들어, '협의 완료 즉시' 또는 '작업 착수 전' 등으로 명확히 할 수 있습니다."},
 {'risk_level': 'low',
  'status': 'pending',
  'title': '검수 완료 기준 및 지급 기한 명확화 필요',
  'clause_no': '5',
 

## 8. Save Accepted HITL Edits

This step applies only findings whose HITL decision status is `accepted`, writes a new document beside the demo outputs, and renders an HTML preview of the edited file.


In [17]:
edited_output_path = None
edited_apply_result = None

pending_status_count = sum(
    finding.status == "pending"
    for finding in (risk_result.findings if risk_result is not None else [])
)
accepted_decision_count = sum(
    decision.get("action") == "accept"
    for decision in globals().get("HITL_DECISIONS", [])
    if isinstance(decision, dict)
)
auto_resumed_for_save = False

if (
    risk_result is not None
    and pending_status_count > 0
    and accepted_decision_count > 0
    and hitl_payload is not None
    and risk_graph is not None
    and risk_thread_config is not None
):
    risk_graph_state = risk_graph.invoke(
        Command(resume={"decisions": HITL_DECISIONS}),
        config=risk_thread_config,
    )
    risk_state = ContractReviewGraphState.model_validate(risk_graph_state)
    risk_result = risk_state.result
    auto_resumed_for_save = True

accepted_findings = [
    finding
    for finding in (risk_result.findings if risk_result is not None else [])
    if finding.status == "accepted" and finding.proposed_edit is not None
]

risk_rank = {"none": 0, "low": 1, "mid": 2, "high": 3, "crit": 4}
accepted_findings_by_target = {}
for finding in accepted_findings:
    target_id = finding.proposed_edit.target_id
    accepted_findings_by_target.setdefault(target_id, []).append(finding)

selected_findings = []
skipped_conflicting_findings = []
for target_id, findings_for_target in accepted_findings_by_target.items():
    ranked = sorted(
        findings_for_target,
        key=lambda finding: (-risk_rank.get(finding.risk_level, 0), finding.finding_id),
    )
    selected_findings.append(ranked[0])
    skipped_conflicting_findings.extend(ranked[1:])

accepted_edits = [finding.proposed_edit for finding in selected_findings]
status_counts = dict(Counter(finding.status for finding in (risk_result.findings if risk_result is not None else [])))
decision_action_counts = dict(Counter(
    decision.get("action")
    for decision in globals().get("HITL_DECISIONS", [])
    if isinstance(decision, dict)
))
conflict_summary = [
    {
        "target_id": finding.proposed_edit.target_id,
        "finding_id": finding.finding_id,
        "risk_level": finding.risk_level,
        "title": finding.title,
        "reason": "같은 문단에 이미 더 우선순위가 높은/앞선 수정안이 선택되어 건너뜀",
    }
    for finding in skipped_conflicting_findings
]

if risk_result is None:
    display({"saved": False, "reason": "먼저 리스크 분석 그래프를 실행하세요."})
elif not accepted_edits:
    display({
        "saved": False,
        "reason": "현재 risk_result에 accepted 상태의 수정안이 없습니다.",
        "auto_resumed_for_save": auto_resumed_for_save,
        "accepted_edit_count": 0,
        "status_counts": status_counts,
        "decision_action_counts": decision_action_counts,
        "next_step": "HITL_DECISIONS에 action='accept'가 있는지 확인하세요. 이미 다른 결정으로 종료된 thread라면 리스크 그래프 셀을 다시 실행해 새 thread를 만든 뒤 다시 저장하세요.",
    })
else:
    edited_output_path = OUTPUT_DIR / f"{SAMPLE_DOC.stem}_accepted_edits{SAMPLE_DOC.suffix}"
    edited_apply_result = apply_document_edits(
        source_path=str(SAMPLE_DOC),
        output_path=str(edited_output_path),
        edits=accepted_edits,
    )
    display({
        "saved": edited_apply_result.ok,
        "auto_resumed_for_save": auto_resumed_for_save,
        "accepted_finding_count": len(accepted_findings),
        "selected_edit_count": len(accepted_edits),
        "skipped_conflicting_edit_count": len(skipped_conflicting_findings),
        "skipped_conflicting_edits": conflict_summary,
        "status_counts": status_counts,
        "edits_applied": edited_apply_result.edits_applied,
        "output_path": str(edited_output_path.relative_to(PROJECT_DIR)),
        "modified_target_ids": edited_apply_result.modified_target_ids,
        "validation_issues": [issue.model_dump(mode="json") for issue in edited_apply_result.validation.issues],
    })


{'saved': True,
 'auto_resumed_for_save': True,
 'accepted_finding_count': 12,
 'selected_edit_count': 11,
 'skipped_conflicting_edit_count': 1,
 'skipped_conflicting_edits': [{'target_id': 'p_52fa12b5f1fa8ace',
   'finding_id': 'clause-11:finding:2',
   'risk_level': 'low',
   'title': '하자 보수 기간의 불명확성',
   'reason': '같은 문단에 이미 더 우선순위가 높은/앞선 수정안이 선택되어 건너뜀'}],
 'status_counts': {'accepted': 12},
 'edits_applied': 11,
 'output_path': 'tests/results/doc_processor_categorization_demo/04_HDC-04_소프트웨어_개발용역_하도급계약서_accepted_edits.docx',
 'modified_target_ids': ['p_4b464a6bb30d2b7b',
  'p_67e6166b1b5c0b03',
  'p_cf4447a915e546f7',
  'p_aa2f77e3ebc544c7',
  'p_6050092e5f0508aa',
  'p_fb9bf524af157a0b',
  'p_380380582a7cf328',
  'p_d9e1129dcbf9a92b',
  'p_8ef0e8427994c595',
  'p_31b3a3d31a808c84',
  'p_52fa12b5f1fa8ace'],
 'validation_issues': []}

In [18]:
edited_preview_result = None
edited_preview_path = None

if edited_apply_result is not None and edited_apply_result.ok and edited_output_path is not None:
    edited_annotations = [
        TextAnnotation(
            target_kind="paragraph",
            target_id=edit.target_id,
            label="수정됨",
            color="#D9EAD3",
            note=edit.reason or "HITL에서 수락된 수정안이 적용되었습니다.",
        )
        for edit in accepted_edits
    ]
    edited_preview_result = render_review_html(
        source_path=str(edited_output_path),
        annotations=edited_annotations,
        title="수정본 미리보기",
    )
    edited_preview_path = OUTPUT_DIR / f"{SAMPLE_DOC.stem}_accepted_edits_preview.html"
    edited_preview_path.write_text(edited_preview_result.html or "", encoding="utf-8")

    display({
        "preview_ok": edited_preview_result.ok,
        "preview_path": str(edited_preview_path.relative_to(PROJECT_DIR)),
        "resolved_annotation_count": len(edited_preview_result.resolved_annotations),
        "validation_issues": [issue.model_dump(mode="json") for issue in edited_preview_result.validation.issues],
    })

    if edited_preview_result.ok and edited_preview_result.html:
        show_html(edited_preview_result.html, height=820)
elif edited_apply_result is not None:
    display({"preview_ok": False, "reason": "수정본 저장이 실패하여 HTML 미리보기를 만들지 않았습니다."})


{'preview_ok': True,
 'preview_path': 'tests/results/doc_processor_categorization_demo/04_HDC-04_소프트웨어_개발용역_하도급계약서_accepted_edits_preview.html',
 'resolved_annotation_count': 11,
 'validation_issues': []}

## 9. Inspect Resolved Annotation Anchors

This small preview is useful when debugging whether the parser category spans resolved to the expected text.
    


In [19]:
resolved_preview = [
    annotation.model_dump(mode="json")
    for annotation in review.resolved_annotations[:20]
]
display(resolved_preview)
    

[{'target_kind': 'paragraph',
  'target_id': 'p_bfa4da5a775f2ffd',
  'selected_text': '소프트웨어 기능개발 용역 하도급계약서',
  'occurrence_index': None,
  'start': 0,
  'end': 20,
  'label': 'Title',
  'color': '#D9EAD3',
  'note': 'category=title'},
 {'target_kind': 'paragraph',
  'target_id': 'p_b137ee595f9f9767',
  'selected_text': 'CRM 모듈 개발 및 API 연동 용역위탁 하도급계약 샘플',
  'occurrence_index': None,
  'start': 0,
  'end': 32,
  'label': 'Preamble',
  'color': '#FFF2CC',
  'note': 'category=preamble'},
 {'target_kind': 'paragraph',
  'target_id': 'p_70a5402160bbf4b7',
  'selected_text': '문서ID: HDC-04',
  'occurrence_index': None,
  'start': 0,
  'end': 12,
  'label': 'Preamble',
  'color': '#FFF2CC',
  'note': 'category=preamble'},
 {'target_kind': 'paragraph',
  'target_id': 'p_181bb4ee0bde7aef',
  'selected_text': '제1조(목적)',
  'occurrence_index': 0,
  'start': 0,
  'end': 7,
  'label': 'Clause Heading',
  'color': '#F4CCCC',
  'note': 'category=clause_heading | clause=1'},
 {'target_kind': 'paragraph'

## Exercise

Try one of these changes and rerun the notebook from step 2:

1. Switch `SAMPLE_DOC` to the second HWPX sample.
2. Toggle `USE_LLM_REVIEW` depending on whether parser LLM credentials are configured.
3. Change one category color in `PARSER_CATEGORY_COLORS` and regenerate the parser category HTML.
4. Set `RUN_RISK_ANALYSIS = True` after the env check passes to render risk highlights from RAG evidence.

Expected result: the category counts, risk findings, and highlighted HTML should change, while the same `TextAnnotation` and `render_review_html(...)` flow stays intact.
